### Imports and loads unique coordinates

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
from pathlib import Path

In [2]:
PBF = "./data/osm/moscow.osm.pbf"

points = gpd.read_parquet("./data/clean.parquet")[["geo_lat", "geo_lon", "geometry"]]
points = points.drop_duplicates(subset=["geo_lat", "geo_lon"])
points = points.to_crs(32637)
points = points.reset_index(drop=True)
points.shape

(46639, 3)

In [3]:
def nearest_distance(points, layer):
    nearest = gpd.sjoin_nearest(points[["geometry"]], layer[["geometry"]], distance_col="distance")
    return nearest.groupby(level=0)["distance"].first()


def sum_in_buffer(points, layer, radius, value):
    buf = points.buffer(radius)
    pairs = gpd.sjoin(gpd.GeoDataFrame(geometry=buf), layer[["geometry"]], predicate="intersects")

    clipped = shapely.intersection(
        buf.loc[pairs.index].values,
        layer.geometry.loc[pairs["index_right"]].values,
    )

    if value == "length":
        pairs["value"] = shapely.length(clipped)
    else:
        pairs["value"] = shapely.area(clipped)

    if "weight" in layer.columns:
        pairs["value"] = pairs["value"] * layer["weight"].loc[pairs["index_right"]].values

    return pairs.groupby(level=0)["value"].sum().reindex(points.index, fill_value=0)

In [4]:
def to_polygons(layer):
    layer = layer.to_crs(32637)

    layer["geometry"] = layer.geometry.make_valid()
    layer = layer.explode(index_parts=False)
    layer = layer[layer.geom_type.isin(["Polygon", "MultiPolygon"])]

    layer = layer[["geometry"]].dissolve()
    layer = layer.explode(index_parts=False).reset_index(drop=True)
    return layer

### Industrial zones

In [5]:
industrial = gpd.read_file(
    PBF,
    layer="multipolygons",
    where="landuse = 'industrial' OR man_made = 'works'",
)
industrial = to_polygons(industrial)
industrial.shape

C:\univer\7_semester\projects\estate_traiding_system\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


(3699, 1)

In [6]:
points["distance_industrial"] = nearest_distance(points, industrial)
points["industrial_area_1000"] = sum_in_buffer(points, industrial, 1000, "area")

### Parks and forests

In [7]:
green = gpd.read_file(
    PBF,
    layer="multipolygons",
    where="""leisure = 'park' OR landuse IN ('forest', 'recreation_ground') OR "natural" = 'wood'""",
)
green = to_polygons(green)
green.shape

(21110, 1)

In [8]:
big_green = green[green.area >= 10_000]

points["distance_park"] = nearest_distance(points, big_green)
points["green_area_500"] = sum_in_buffer(points, green, 500, "area")
points["green_area_1000"] = sum_in_buffer(points, green, 1000, "area")

### Water

In [9]:
rivers = gpd.read_file(PBF, layer="lines", where="waterway = 'river'")
rivers = rivers.to_crs(32637)

water = gpd.read_file(PBF, layer="multipolygons", where=""""natural" = 'water'""")
water = to_polygons(water)

water = water[water.area >= 10_000]

rivers.shape, water.shape

((1549, 11), (677, 1))

In [10]:
all_water = pd.concat([rivers[["geometry"]], water[["geometry"]]], ignore_index=True)
all_water = gpd.GeoDataFrame(all_water, geometry="geometry", crs=32637)

points["distance_water"] = nearest_distance(points, all_water)

### Save

In [13]:
features = pd.DataFrame(points.drop(columns="geometry"))

assert features.shape[0] == features[["geo_lat", "geo_lon"]].drop_duplicates().shape[0]
assert features.isna().sum().sum() == 0

Path("./data/geo").mkdir(exist_ok=True)
features.to_parquet("./data/geo/env.parquet", index=False)


,geo_lat,geo_lon,distance_industrial,industrial_area_1000,distance_park,green_area_500,green_area_1000,distance_water
0,55.738876,37.825370,744.226392,79072.887882,281.624376,80170.756372,812161.732507,188.466373
1,55.798921,37.738090,146.931108,220047.100454,141.979279,125788.669607,290275.682205,311.243865
2,55.655307,37.614605,594.902473,312934.893308,234.823725,16147.594210,103343.832179,1653.798179
3,55.701330,37.507412,602.495220,58400.553774,418.546838,47708.485147,173506.457006,1071.446720
4,55.730785,37.631424,131.690441,138277.983584,334.077438,37425.248222,123039.194341,704.353144
